# Database

In [2]:
# download the ds
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-05-08 14:24:30--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8002::154, 2606:50c0:8003::154, 2606:50c0:8000::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8002::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  3.24MB/s    in 0.3s    

2026-05-08 14:24:30 (3.24 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [3]:
# read the ds
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# examine first N words in the ds
print(text[:300])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [4]:
# our vocabulary 
chars = sorted(list(set(text)))  # sorted list of chars
vocab_size = len(chars)

print(chars)
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


In [5]:
# look-up. mapping from characters to integers (strings & integers)
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

# encoder: take a string, output a list of integers
encode = lambda s: [stoi[c] for c in s]
# decoder: take a list of integers, output a string
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


__________________
Comment:
- enumerate() - if chars is this list ['h', 'e', 'l', 'o'], enumerate gives you pairs of (position, character): (0, 'h'), (1, 'e'), (2, 'l'), (3, 'o')
- for i,ch - for every pair (e.g., (0, 'h')), call the number i and the letter ch
- {ch:i} - create a dictionary (key = character, value = number)
- lambda s: [] - create a small function that takes s (a string) and create a list
- stoi[c] for c in s - in that list, apply stoi to each character in a string
- ''.join() - take all those characters and glue them together into one string
__________________

In [6]:
# encode ds & store it into a torch.tensor
import torch

data = torch.tensor(encode(text), dtype = torch.long)

print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [7]:
# split up data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# Input and Output

When we train the Transformer, we do not feed the whole text at once (computationaly expensive). 
Instead, we sample random little chunks out of the set and train on chunks.
These chunks have block_size = the maximum length of the chunk, or context length.
And they have batch_size = how many chunks will be fed in parallel.

______________

**What is (B,T,C):**  
- B - Batch_size - How many sequences are processed in parallel
- T - Time / Sequence_length / Block_size - How many tokens are in one sequence
- C - Channels / Classes / Vocab_size - Number of all unique tokens in vocabulary

Why "Time"(T)?  
Even though we are dealing with text, we call this dimension "Time" because language models are sequential — they process tokens one after another, like time steps. This naming comes from Recurrent Neural Networks (RNNs) and is still used in Transformers and all modern language models. T=1 → only one token, T=8 → 8 tokens in context, T=1024 → long context (like in GPT models)

Why "Channels"(C)?  
"Channels" is a term borrowed from Computer Vision. In images: channels = RGB (3 channels). In language: channels = vocabulary size. Each position has a vector with one number (logit) for every possible token in the vocabulary. That’s why C = vocab_size.
______________

In [8]:
# example of one chunk (size 8, first chars):
block_size = 8
print("One chunk:", train_data[:block_size+1])

# visualization of how it works:
x = train_data[:block_size] # input (first 8 elements)
y = train_data[1:block_size+1] # output (is shifted to 1 element)
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

One chunk: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])
when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [9]:
# now let's combine random chunks into batches of size 4
# and prepare the correct output, also the same size.
# (if input is word index x1, output is word index x1+1. Similarly, if x1+x2+x3, then output is )

torch.manual_seed(1337)

batch_size = 4
block_size = 8

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data  # if there is a split, use train data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

# xb - input to the transformer, yb - correct output
xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)
print('----')

# explanation again:
for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

# Bigram model
We will start with the simplest NN, the bigram model. No attention. only forward pass and prediction from the last token.

In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

# bigram model is available in torch.nn module
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size): # vocab_size=len(chars)=65
        super().__init__()  # calls the parent class (nn.Module) initializer. Required in PyTorch when you inherit from nn.Module.
        # create embedding layer (lookup table of size vocab_size * vocab_size)
        # each 65 row is one token - a vector of 65 embedded values (logits)
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    # get predictions
    def forward(self, idx, targets=None):  # idx = input, targets = output
        # idx and targets are both 2D (B,T) tensors of integers (B batch=batch_size, T time=block_size) like [4,8] (see above inputs and targets example of shape 4*8)
        # now look up the embedding for every token in idx:
        logits = self.token_embedding_table(idx) # embedding layer turns every token ID into a vector of length C (vocab_size). So now it is 3D: (B,T,C) batch=batch_size, time=block_size, channels=vocab_size, like [4,8,65]
        
        # loss
        if targets == None: # during generation we only want logits, no loss needed
            loss = None
        else:  # during training we compute loss.
            # but F.cross_entropy() expects 2D input for preds and 1D for targets. Our target is (B,T), and logits are (B,T,C)...
            # Let's flatten our B and T dimensions into one big dimension:
            B, T, C = logits.shape  # unpack the shape: B=4, T=8, C=65
            logits = logits.view(B*T, C)  # reshape the 3D tensor into 2D: B*T=4*8=24 >> [24,65]
            targets = targets.view(B*T)  # reshape the 2D tensor into 1D: B*T=4*8=24 >> [24]
            # do not worry, the logic is safe: 1st row in logits (1st token - vector of size 65) corresponds to 1st row in targets (correct predict token)
            loss = F.cross_entropy(logits, targets)  # take each row in logits, compare it with the corresponding correct index in targets, return a single number (the average loss)
        return logits, loss  # return loss only if targets were provided

    # now let's generate smth from the model
    def generate(self, idx, max_new_tokens):  # again, idx is 2D (B, T) array, like [4,8]; max_new_tokens - how many new tokens to generate.
        # goal is to take (B,T) and generate (B,T)+1,+2,+3...as many as we want max_new_tokens
        for _ in range(max_new_tokens):  # loop that appends one token at a time
            # get predictions:
            logits, loss = self(idx)  # call forward. Since we don't pass targets, loss will be None (we don't need it at this stage).
            # for now, we are not interested in attention. We want only how the last token influences the target.
            # therefore, for each batch (size 4), we need only the last token (last token's logits) and get rid of all tokens before it (all 7 out of 8) = overall 4 last tokens' logits per batch (size 4):
            logits = logits[:, -1, :] # Python slicing takes: all 4 batches, only last tokens, all 65 logits of them. (B,T,C) becomes (B, C) >> [4,65] - 4 tokens (since batch_size 4) with 65 logits each.
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # dim=-1 means apply softmax along the last dimension (the vocabulary dimension). probs are still (B, C)
            # sample one random token from the softmax probability distribution
            idx_next = torch.multinomial(probs, num_samples=1) # multinomial does weighted random sampling (higher probability tokens are more likely to be chosen) and returns one token index per sequence: (B, 1) ([4,1] since batch_size=4)
            # take chosen token and append it to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # torch.cat(..., dim=1) concatenates along the T dimension (B, T+1)
        return idx  # result after the loop >> idx contains the original tokens + all newly generated ones

# testing forward pass in training)
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)  # xb = input batch, yb = target batch from above
print(logits.shape)
print(loss)

# testing generation 
idx = torch.zeros((1, 1), dtype=torch.long)  # create a starting context: a single sequence containing only token 0
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))  # [0] selects the first sequence in the batch; tolist() converts PyTorch tensor into a list

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


# Training

In [21]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [23]:
# training

batch_size = 32  # let's use new batch_size, bigger than 4

for steps in range(10000): # increase number of steps for good results...
    # sample a batch of data
    xb, yb = get_batch('train')
    # evaluate the loss
    logits, loss = m(xb, yb)
    # zero our gradients from the previous step
    optimizer.zero_grad(set_to_none=True)
    # get gradients for all parameters
    loss.backward()
    # use those gradients to update parameters
    optimizer.step()
    
print(loss.item())

2.355031728744507


In [26]:
# check generation
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=500)[0].tolist()))


HAnadire ckid t ait Makes, INTher emy s isuthiar, sherthilouce rbis orilld: bel bry whoulou GHEOfoul ve I hes aikso' heretr hima Ghantheraritheashenks cirund otlveileiuge bu peajord t'domy boinope l, s! bofe, 't myouroupous y centirbrgstoliterinalend tslisthins me f cheme, s,
ARCKApeaghof usthe pt
arere othat wis ativer ers thes;

K: e mand ary pendo, t:
LAnead ir;
ICK: n
HAUKit core.
Fothe fomat, p'lt py t Tu w'str usthitherorrte, ber fe sath t ofo IO:
Nay d CHee,
A:
Thene;


Herst yey h-bent c
